# multimodal-query-demo — one-click reproduce

This notebook clones the repository, installs a pinned Rust toolchain, runs the full test suite, regenerates the benchmark report, renders the five figures, and (optionally) rebuilds the paper PDF.

**Expected runtime:** ~8–12 minutes on a Colab CPU runtime. The first `cargo test --release` cell is the single longest step (~3–6 min of silent compilation on a cold VM). Every subsequent cell adds per-step progress markers so you can tell it hasn't frozen.

**What you get at the end:** `out/bench.json`, `paper/figures/*.png` (five PNGs), reproducible bit-for-bit from a clean VM given the same toolchain version.

**What this notebook does not claim:** that the numbers generalise to any other machine. Bench latencies are single-host wall-clock, and absolute values will differ on Colab CPU vs. the author's workstation. The *shapes* (ratios, slopes, pruning counts) are what reproduce.

## 1. Clone the repository

In [ ]:
%%bash
set -eu
echo "[$(date +%H:%M:%S)] clone…"
if [ ! -d multimodal-query-demo ]; then
  git clone --depth 1 https://github.com/infinityabundance/multimodal-query-demo.git
else
  echo "already cloned"
fi
cd multimodal-query-demo && git log -1 --oneline
echo "[$(date +%H:%M:%S)] clone done"

## 2. Install Rust (pinned to workspace MSRV 1.85)

Idempotent: if `cargo` is already on PATH, the download is skipped.

In [ ]:
%%bash
set -eu
echo "[$(date +%H:%M:%S)] rust install…"
SECONDS=0
if [ ! -x "$HOME/.cargo/bin/cargo" ] && ! command -v cargo >/dev/null; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.85.0 --profile minimal
else
  echo "cargo already present"
fi
export PATH="$HOME/.cargo/bin:$PATH"
cargo --version && rustc --version
echo "[$(date +%H:%M:%S)] rust install done in ${SECONDS}s"

## 3. Warm the release build cache

Compiles the whole workspace once so the subsequent `cargo test`, `cargo run`, and `cargo bench` cells just execute instead of re-checking and rebuilding. Takes **~3–6 minutes** on a cold Colab VM and is the single longest cell in the notebook — cargo streams per-crate compilation lines so you can see progress.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
echo "[$(date +%H:%M:%S)] cargo build --workspace --release --tests (first build: 3-6 min)…"
SECONDS=0
cargo build --workspace --release --tests
echo "[$(date +%H:%M:%S)] build done in ${SECONDS}s"

## 4. Run the workspace test suite

47 tests including `non_claim_lock` (parses the paper's non-claims against the Rust constant), `deterministic_replay` (SHA-256 fingerprint of a seeded proximity output), and `concurrent_snapshots` (reader/writer stress). With the cache warm from §3 this should take **under 30 seconds**.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
echo "[$(date +%H:%M:%S)] cargo test --workspace --release…"
SECONDS=0
cargo test --workspace --release
echo "[$(date +%H:%M:%S)] tests done in ${SECONDS}s"

## 5. Exercise every engine path

Hits every `mqd` subcommand end-to-end. `--explain` dumps the `QueryPlan` (partition counts, pruning counts, scanned rows) so the engine's work is visible, not just the wall clock. Seven sub-steps, each with its own timestamped header.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
mkdir -p data out paper/figures
step() { echo; echo "=== [$(date +%H:%M:%S)] $* ==="; }

step "[1/7] generate (seed=42, agents=8, ticks=10000) → data/run.jsonl"
cargo run --quiet -p mqd-cli --release -- generate --seed 42 --agents 8 --ticks 10000 --out data/run.jsonl

step "[2/7] ingest"
cargo run --quiet -p mqd-cli --release -- ingest --input data/run.jsonl --batch-rows 4096

step "[3/7] query latest-at --explain"
cargo run --quiet -p mqd-cli --release -- query latest-at --input data/run.jsonl --at 25000000000 --explain

step "[4/7] query range --explain"
cargo run --quiet -p mqd-cli --release -- query range --input data/run.jsonl --start 0 --end 10000000000 --entity /agent/0 --kind transform3d --explain

step "[5/7] op proximity (first 20 lines)"
cargo run --quiet -p mqd-cli --release -- op proximity --input data/run.jsonl --at 25000000000 --radius 2.5 --json | head -20

step "[6/7] op speed (first 20 lines)"
cargo run --quiet -p mqd-cli --release -- op speed --input data/run.jsonl --entity /agent/0 --json | head -20

step "[7/7] non-claims charter"
cargo run --quiet -p mqd-cli --release -- non-claims

step "all engine paths done"

## 6. Bench (51 samples × 17 scenarios) and render figures

`bench` produces a single JSON file with p50/p95/p99 per suite × scale. `figs` reads that JSON and writes five PNGs: `ingest_latency.png`, `latest_at_latency.png`, `range_scan_latency.png`, `proximity_latency.png`, `speed_latency.png`. Each carries a p50 solid line, p95 dashed line, and (for proximity / speed) a reference-complexity curve.

The bench cell streams a line per scenario as it finishes — expect **~2–4 minutes** total on Colab. If you want a quicker smoke-test, change `--samples 51` to `--samples 11`.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
echo "=== [$(date +%H:%M:%S)] bench start (17 scenarios × 51 samples) ==="
SECONDS=0
cargo run --quiet -p mqd-cli --release -- bench --out out/bench.json --samples 51
echo "=== [$(date +%H:%M:%S)] bench done in ${SECONDS}s ==="

echo
echo "=== [$(date +%H:%M:%S)] rendering figures ==="
cargo run --quiet -p mqd-cli --release -- figs --from out/bench.json --out paper/figures/
ls -la paper/figures/
echo "=== [$(date +%H:%M:%S)] figs done ==="

## 7. Inspect the bench report

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('multimodal-query-demo/out/bench.json').read_text())
print(f"version={report['version']}  machine={report['machine']}  runs={len(report['runs'])}")
for r in report['runs']:
    print(f"  {r['suite']:12s} scale={r['scale']:10s} x={r['x']:>8.1f}  p50={r['p50_us']:>9.1f}µs  p95={r['p95_us']:>9.1f}µs  p99={r['p99_us']:>9.1f}µs  returned={r['returned_rows']}")

## 8. Render the figures inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for png in sorted(Path('multimodal-query-demo/paper/figures').glob('*_latency.png')):
    print(png.name)
    display(Image(filename=str(png)))

## Done

What you should have:

- `multimodal-query-demo/out/bench.json` — single-file bench report (p50/p95/p99 per suite × scale).
- `multimodal-query-demo/paper/figures/*.png` — five figures referenced by the paper.

If any cell in §4 (tests) failed, the artefact is broken on this runtime; please report the failing test name.